# LIONBOT v3 — ต่อยอดจาก v2

เทรนต่อจากเช็คพอยต์ LoRA เดิมของ **typhoon-lionbot-v2** (ถ้าหาเจอใน Google Drive) แล้ว export เป็น GGUF สำหรับ Ollama ในชื่อ `typhoon-lionbot-v3`

**สำคัญ:** `phakin.gguf` / `typhoon2.1-gemma3-4b.Q4_K_M.gguf` ที่มีอยู่ในเครื่อง เป็นไฟล์ GGUF ที่ quantize แล้ว **เทรนต่อจากไฟล์นี้โดยตรงไม่ได้** — ต้องใช้เช็คพอยต์ LoRA/HF ต้นทางที่ยังไม่ถูก merge+quantize เท่านั้น สมุดบันทึกนี้จะ**ค้นหาเช็คพอยต์นั้นใน Drive ให้อัตโนมัติ** ถ้าไม่เจอ จะแจ้งเตือนและ fallback ไปเทรนใหม่จาก base model เดิม (`scb10x/typhoon2.1-gemma3-4b`) แทน — ยังได้โมเดลใหม่ที่ใช้งานได้ แต่จะไม่ใช่การ "ต่อยอด" ของจริง

**ตั้ง Runtime เป็น T4 GPU ก่อนรัน:** Runtime → Change runtime type → T4 GPU

## 1. ติดตั้ง Unsloth + เช็ค GPU

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir "git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี GPU — ไปที่ Runtime > Change runtime type > T4 GPU ก่อน")

## 2. เชื่อม Google Drive + ค้นหาเช็คพอยต์เดิม

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ค้นหา adapter_config.json (เช็คพอยต์ LoRA ของ v2) ทั่ว Drive
# รอบเทรน v2 อาจไม่ได้เซฟ adapter แยกไว้ (เซฟแค่ GGUF ที่ merge แล้ว) — ถ้าหาไม่เจอ cell ถัดไปจะแจ้งเตือนและ fallback ให้เอง
import glob

candidate_dirs = glob.glob('/content/drive/MyDrive/**/adapter_config.json', recursive=True)
PREV_ADAPTER_PATH = None

if candidate_dirs:
    PREV_ADAPTER_PATH = candidate_dirs[0].rsplit('/', 1)[0]
    print(f"✅ เจอเช็คพอยต์ LoRA เดิม: {PREV_ADAPTER_PATH}")
    if len(candidate_dirs) > 1:
        print("⚠️ เจอมากกว่า 1 ที่ — ใช้อันแรก ถ้าอยากใช้อันอื่นแก้ PREV_ADAPTER_PATH เอง:")
        for c in candidate_dirs:
            print("  -", c.rsplit('/', 1)[0])
else:
    print("⚠️ ไม่เจอเช็คพอยต์ LoRA เดิมใน Drive (มีแต่ GGUF ที่ merge แล้ว ต่อยอดโดยตรงไม่ได้)")
    print("   จะ fallback ไปเทรนใหม่จาก base model scb10x/typhoon2.1-gemma3-4b แทน (เหมือนตอนเทรน v2)")

## 3. โหลดโมเดล (ต่อยอดถ้าเจอ adapter / เริ่มใหม่ถ้าไม่เจอ)

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "scb10x/typhoon2.1-gemma3-4b"
MAX_SEQ_LENGTH = 2048

if PREV_ADAPTER_PATH:
    # ต่อยอด: โหลด adapter เดิมมาเทรนต่อ
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=PREV_ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    print(f"โหลดต่อยอดจาก {PREV_ADAPTER_PATH} สำเร็จ")
else:
    # fallback: เริ่มใหม่จาก base model เดิม (เหมือน v2)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    print(f"เริ่ม LoRA ใหม่จาก {BASE_MODEL}")

## 4. อัปโหลด dataset.jsonl
อัปโหลดไฟล์ `my-app/training/dataset/dataset.jsonl` จากเครื่อง (240 ตัวอย่างเดิม) — ถ้ามีชุดข้อมูลใหม่/เพิ่มเติม อัปโหลดไฟล์นั้นแทนหรือรวมกันก่อนอัปโหลดได้เลย

In [ ]:
from google.colab import files
uploaded = files.upload()  # เลือก dataset.jsonl
DATASET_FILE = list(uploaded.keys())[0]
print("ใช้ไฟล์:", DATASET_FILE)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"จำนวนตัวอย่าง: {len(dataset)}")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(dataset[0]["text"][:400])

## 5. เทรน

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)
trainer_stats = trainer.train()

## 6. เซฟ adapter ลง Drive (สำคัญ — กันปัญหานี้ไม่ให้เกิดซ้ำตอนเทรน v4)

In [ ]:
ADAPTER_OUT = "typhoon-lionbot-v3-adapter"
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

import shutil
DRIVE_ADAPTER_DEST = f"/content/drive/MyDrive/{ADAPTER_OUT}"
shutil.copytree(ADAPTER_OUT, DRIVE_ADAPTER_DEST, dirs_exist_ok=True)
print(f"เซฟ adapter ไว้ที่ {DRIVE_ADAPTER_DEST} แล้ว — รอบหน้า (v4) notebook นี้จะหาเจอเองอัตโนมัติ")

## 7. แปลงเป็น GGUF (q4_k_m) สำหรับ Ollama

In [ ]:
GGUF_NAME = "typhoon-lionbot-v3-gguf"
model.save_pretrained_gguf(GGUF_NAME, tokenizer, quantization_method="q4_k_m")

In [ ]:
# path ที่ Unsloth สร้างจริงบางทีต่างจากชื่อโฟลเดอร์ที่ตั้งไว้ (เจอปัญหานี้ตอนเทรน v2) — ค้นด้วย glob กันพลาด
import glob, shutil, os

gguf_files = glob.glob(f"**/*.gguf", recursive=True)
gguf_files = [f for f in gguf_files if "q4_k_m" in f.lower() or "Q4_K_M" in f]
if not gguf_files:
    gguf_files = glob.glob("**/*.gguf", recursive=True)

print("เจอไฟล์ GGUF:", gguf_files)
assert gguf_files, "ไม่เจอไฟล์ .gguf — เช็ค cell ก่อนหน้าว่า export สำเร็จหรือไม่"

SRC_GGUF = gguf_files[0]
DRIVE_GGUF_DIR = "/content/drive/MyDrive/typhoon-lionbot-v3-gguf"
os.makedirs(DRIVE_GGUF_DIR, exist_ok=True)
DEST_GGUF = os.path.join(DRIVE_GGUF_DIR, "typhoon-lionbot-v3.Q4_K_M.gguf")
shutil.copy(SRC_GGUF, DEST_GGUF)
print(f"✅ คัดลอกไปที่ {DEST_GGUF} แล้ว")

## ขั้นตอนที่เหลือ (ทำในเครื่อง ไม่ใช่ใน Colab)

1. ดาวน์โหลด `typhoon-lionbot-v3.Q4_K_M.gguf` จาก Google Drive มาไว้ที่ `my-app/backend/`
2. แก้ `my-app/backend/Modelfile.txt`:
   ```
   FROM ./typhoon-lionbot-v3.Q4_K_M.gguf
   PARAMETER temperature 0.7
   SYSTEM คุณคือ LIONBOT ผู้ช่วยอัจฉริยะด้านหลักสูตรวิศวกรรมไฟฟ้าและอิเล็กทรอนิกส์
   ```
3. รัน `ollama create typhoon-lionbot-v3 -f Modelfile`
4. เพิ่ม `"typhoon-lionbot-v3": "Typhoon LIONBOT v3 (Fine-Tuned)"` เข้า `AVAILABLE_MODELS` ใน `my-app/backend/server.py` และปรับ `DEFAULT_MODEL` ถ้าต้องการตั้งเป็นค่าเริ่มต้น
5. ทดสอบผ่านหน้าเว็บแชท